# Quantile regression: projected-LP Gurobi algorithm comparison

Research question: how does the projected LP's algorithm affect solve time,
retrieval time, and solution quality when the original solver, instance, and T
are held fixed?

The source notebook `quantile_regression_gurobi_alg_comparison.ipynb` is preserved.
This notebook copies its helper functions and timed Gurobi solve without changing
the QR formulation or retrieval mathematics.

- **Once:** generate D/b/A/bounds, solve the original LP with Automatic, generate
  T, and form TA/Tb. Original Automatic is the fixed reference for all ratios.
- **Once per projected algorithm:** build a fresh model, optimize, extract the
  solution, then run Dykstra on that solution. Compare Automatic, primal simplex,
  dual simplex, barrier with default crossover, and barrier without crossover.
- `REPEATS = 1` follows the user's latest time constraint. Increasing it repeats
  the original Automatic baseline and every projected method, using fresh models
  and rotating projected execution order. The instance and T stay unchanged.
  With one run, timings are single observations and standard deviations are
  unavailable; they are displayed as `n/a`, not zero.

Time uses `perf_counter()` wall seconds. Model creation/solve/extraction is
included; license initialization, shared data generation, and quality checks are
excluded. T generation and TA/Tb are measured once, then their costs are included
in every complete projected pipeline. No models, bases, or solutions are reused
as warm starts. All runs are sequential and logs are quiet by default.

**Retrieval limitation:** the existing Dykstra implementation is intentionally
retained, including the previously identified case where a bound-projected point
is not assigned back to x when its equality error is small. Its stopping label
does not establish feasibility. Every returned point is checked independently;
infeasible points are flagged and have no valid feasible objective-gap claim.
The raw objective difference is still printed explicitly as descriptive.

Liberti et al. (local `Liberti et al.pdf`, Sections 3.7–3.8 and Table 5) use CPLEX
Automatic for the original LP and HiGHS revised simplex for projected QR. Their
m=5000, epsilon=0.2 time ratio is 0.19, averaged under a different timing scope
and environment. Here both solvers are Gurobi, so this is not a replication of
their cross-solver benchmark. It isolates projected algorithm settings on one
instance and one fixed projection. A single run does not establish a stable
ranking across timings, instances, or projections.

[Gurobi Method/Crossover parameters](https://docs.gurobi.com/projects/optimizer/en/current/reference/parameters.html)
and [ConcurrentWinMethod](https://docs.gurobi.com/projects/optimizer/en/current/reference/attributes/model.html#concurrentwinmethod)
describe the solver settings and reported concurrent winners.


In [1]:
import time
import platform
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse.linalg import lsmr
import gurobipy as gp
from gurobipy import GRB

# Reference instance and projected/retrieval settings.
M = 5000
P = 399
DENSITY = 0.8
TAU = 0.2
EPSILON = 0.2
DATA_SEED = 42
PROJECTION_SEED = 42
MAX_ITER = 30
DELTA = 0.01

REPEATS = 1  # User-selected time budget: one baseline solve and one run per TLP algorithm.
THREADS = 0  # Gurobi automatic allocation, as in the reference.
SOLVER_SEED = 0  # Gurobi default; fixed for all models.
OUTPUT_FLAG = 0  # Quiet output; set to 1 for Gurobi logs (may affect wall time).

PAPER_RATIO = 0.19
OBSERVED_RATIO = 0.334
ALGORITHMS = [
    ("Automatic", -1, -1),
    ("Primal simplex", 0, -1),
    ("Dual simplex", 1, -1),
    ("Barrier default crossover", 2, -1),
    ("Barrier without crossover", 2, 0),
]
assert REPEATS >= 1


In [2]:
def generate_quantile_instance(
    m: int,
    p: int,
    density: float = 0.8,
    seed: int | None = None,
):
    """
    Generate one synthetic Quantile Regression instance.

    Parameters
    ----------
    m : int
        Number of observations / records.

    p : int
        Number of explanatory variables (columns of D).

    density : float, default=0.8
        Target proportion of nonzero entries in D.

    seed : int or None, default=None
        Random seed for reproducibility.

    Returns
    -------
    D : ndarray, shape (m, p)
        Design matrix.

    b : ndarray, shape (m,)
        Dependent variable.
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # check input values
    if m <= 0:
        raise ValueError("m must be positive")

    if p <= 0:
        raise ValueError("p must be positive")

    if not 0.0 <= density <= 1.0:
        raise ValueError("density must lie between 0 and 1")

    # initialize D as an all-zero matrix
    D = np.zeros((m, p), dtype=float)

    # calculate the total number of entries
    n_total = m * p

    # calculate the number of nonzero entries
    n_nonzero = int(round(density * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # fill the selected positions with values sampled from U[-1, 1]
    D.flat[nonzero_indices] = rng.uniform(-1.0, 1.0, size=n_nonzero)

    # generate the dependent variable from U[-1, 1]
    b = rng.uniform(-1.0, 1.0, size=m)

    return D, b


def build_quantile_lp(
    D: np.ndarray,
    b: np.ndarray,
    tau: float = 0.2,
):
    """
    Build the LP formulation of a quantile regression instance.

    Parameters
    ----------
    D : ndarray, shape (m, p)
        Design matrix

    b : ndarray, shape (m,)
        Dependent variable

    tau : float, default=0.2
        Quantile level

    Returns
    -------
    A : sparse matrix, shape (m, p + 2m)
        Equality constraint matrix

    c : ndarray, shape (p + 2m,)
        Objective coefficient vector

    bounds : list
        Variable bounds for beta, u_plus, and u_minus
    """

    # get the dimensions of D
    m, p = D.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError("b must have shape (m,)")

    # check the quantile level
    if not 0.0 < tau < 1.0:
        raise ValueError("tau must lie strictly between 0 and 1")

    # convert D to sparse format
    D_sparse = sparse.csr_matrix(D)

    # construct a sparse identity matrix
    I = sparse.eye(m, format="csr")

    # construct A = [D | I | -I]
    A = sparse.hstack((D_sparse, I, -I), format="csr")

    # construct the objective coefficient vector
    c = np.concatenate(
        (np.zeros(p), tau * np.ones(m), (1.0 - tau) * np.ones(m))
    )

    return A, c


def generate_projection_matrix(
    A,
    k: int,
    seed: int | None = None,
):
    """
    Generate a sparse random projection matrix T.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Constraint matrix

    k : int
        Projected dimension

    seed : int or None, default=None
        Random seed for reproducibility

    Returns
    -------
    T : sparse matrix, shape (k, m)
        Sparse random projection matrix
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # get the number of original constraints
    m = A.shape[0]

    # check the projected dimension
    if not 0 < k < m:
        raise ValueError("k must lie strictly between 0 and m")

    # calculate the density of A
    density_A = A.nnz / (A.shape[0] * A.shape[1])

    # set the density of T
    sigma = density_A / 2.0

    # calculate the total number of entries in T
    n_total = k * m

    # calculate the number of nonzero entries in T
    n_nonzero = int(round(sigma * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # convert the flat indices to row and column indices
    rows = nonzero_indices // m
    cols = nonzero_indices % m

    # calculate the standard deviation of nonzero entries
    std = 1.0 / np.sqrt(k * sigma)

    # sample the nonzero entries from the Gaussian distribution
    values = rng.normal(loc=0.0, scale=std, size=n_nonzero)

    # construct the sparse projection matrix
    T = sparse.csr_matrix((values, (rows, cols)), shape=(k, m))

    return T


def project_lp(
    A,
    b: np.ndarray,
    T,
):
    """
    Project the equality constraints of an LP.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Original constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    T : sparse matrix, shape (k, m)
        Random projection matrix

    Returns
    -------
    A_projected : sparse matrix, shape (k, n)
        Projected constraint matrix

    b_projected : ndarray, shape (k,)
        Projected right-hand side vector
    """

    # check the dimensions of A and b
    if b.shape != (A.shape[0],):
        raise ValueError("b must have shape (m,)")

    # check the dimensions of T and A
    if T.shape[1] != A.shape[0]:
        raise ValueError("T and A have incompatible dimensions")

    # project the constraint matrix
    A_projected = T @ A

    # project the right-hand side vector
    b_projected = T @ b

    return A_projected, b_projected


def build_quantile_bounds(
    m: int,
    p: int,
):
    """
    Build the variable bounds for the quantile regression LP.

    Parameters
    ----------
    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    lb : ndarray, shape (p + 2m,)
        Lower bounds

    ub : ndarray, shape (p + 2m,)
        Upper bounds
    """

    # construct the lower bounds
    lb = np.concatenate((np.full(p, -np.inf), np.zeros(m), np.zeros(m)))

    # construct the upper bounds
    ub = np.full(p + 2 * m, GRB.INFINITY)

    return lb, ub


def solve_projection_system(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    x_check: np.ndarray,
):
    """
    Approximate the projection of x0 onto A x = b using LSMR.
    """

    # solve A x_hat = b + A x0
    x_hat = lsmr(A, b + A @ x0)[0]

    # construct a second point approximately satisfying A x = b
    x_prime = x_hat - x0

    # construct the direction from x_prime to x_check
    direction = x_check - x_prime

    # handle the degenerate case
    if direction @ direction == 0.0:
        return x_prime

    # calculate the projection coefficient
    lambda_value = (direction @ (x0 - x_prime)) / (direction @ direction)

    # project x0 onto the line through x_prime and x_check
    x_dagger = x_prime + lambda_value * direction

    return x_dagger


def bound_error_sum(
    x: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Calculate the sum of bound violations used in the authors' public code.
    """

    # calculate lower-bound violations
    lower_error = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_error = np.maximum(x - ub, 0.0)

    # return the sum of all bound violations
    return float(np.sum(lower_error) + np.sum(upper_error))


def retrieval_feasibility(
    x,
    A,
    b,
    lb,
    ub,
    tol=0.01,
):
    """
    Check feasibility using the error measure and tolerance
    used in the authors' public retrieval code.
    """

    # calculate the L2 equality residual
    equality_error = float(np.linalg.norm(A @ x - b))

    # calculate the sum of all bound violations
    bound_error = bound_error_sum(
        x,
        lb,
        ub,
    )

    # apply the authors' combined error criterion
    return bound_error + equality_error <= 2.0 * tol


def retrieval_start(
    x_projected: np.ndarray,
    A,
    b: np.ndarray,
):
    """
    Apply the LSMR retrieval step used before the authors' APM code.
    """

    # calculate the original equality residual of the projected LP solution
    residual = b - A @ x_projected

    # approximately solve A x_part = b - A x_projected
    x_part = lsmr(A, residual)[0]

    # construct the starting point used by the authors' APM code
    x_start = x_projected + x_part

    return x_start


def evaluate_retrieval(
    x: np.ndarray,
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Evaluate a retrieved solution.
    """

    # calculate the objective value
    objective_value = c @ x

    # calculate the equality constraint residual
    equality_residual = np.linalg.norm(A @ x - b)

    # calculate lower-bound violations
    lower_violation = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_violation = np.zeros_like(x)

    finite_upper = np.isfinite(ub)

    upper_violation[finite_upper] = np.maximum(
        x[finite_upper] - ub[finite_upper], 0.0
    )

    # calculate the maximum bound violation
    bound_violation = max(np.max(lower_violation), np.max(upper_violation))

    return (objective_value, equality_residual, bound_violation)


def evaluate_errors(
    x: np.ndarray,
    A,
    b: np.ndarray,
    m: int,
    p: int,
):
    """
    Calculate the average feasibility errors reported in Table 5.

    Parameters
    ----------
    x : ndarray, shape (p + 2m,)
        Retrieved quantile regression solution

    A : sparse matrix, shape (m, p + 2m)
        Original equality constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    average_inequality_error : float
        Average nonnegativity violation of u_plus and u_minus

    average_equality_error : float
        Average absolute violation of A x = b
    """

    # check the dimension of the retrieved solution
    if x.shape != (p + 2 * m,):
        raise ValueError(f"x must have shape {(p + 2 * m,)}")

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError(f"b must have shape {(m,)}")

    # extract the positive residual variables
    u_plus = x[p : p + m]

    # extract the negative residual variables
    u_minus = x[p + m : p + 2 * m]

    # calculate violations of u_plus >= 0
    u_plus_violation = np.maximum(-u_plus, 0.0)

    # calculate violations of u_minus >= 0
    u_minus_violation = np.maximum(-u_minus, 0.0)

    # calculate the average inequality error
    average_inequality_error = (
        np.sum(u_plus_violation) + np.sum(u_minus_violation)
    ) / m

    # calculate the original equality residual vector
    equality_error = A @ x - b

    # calculate the average equality error
    average_equality_error = np.mean(np.abs(equality_error))

    return (average_inequality_error, average_equality_error)


def retrieve_dykstra(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
):
    """
    Apply the Dykstra APM logic from the authors' public code.
    """

    # first apply the authors' LSMR retrieval step
    x = retrieval_start(x0, A, b)

    # calculate one approximate solution of A x = b
    x_check = lsmr(A, b)[0]

    # initialize the two Dykstra correction vectors
    p = np.zeros_like(x)
    q = np.zeros_like(x)

    # initialize the bound-projected point
    y = np.zeros_like(x)

    # use the authors' default status until another condition is met
    status = "iteration limit"

    for iteration in range(1, max_iter + 1):
        # store the point at the beginning of the iteration
        x_previous = x.copy()

        # calculate the bound-error sum before the bound projection
        bound_error = bound_error_sum(x, lb, ub)

        # the authors stop immediately if there is no material bound error
        if bound_error <= delta:
            status = "no bound error"
            return x_previous, iteration, status

        # project x + p onto the bound set
        y = np.clip(x + p, lb, ub)

        # update the first Dykstra correction vector
        p = x + p - y

        # calculate the equality residual after the bound projection
        equality_error = float(np.linalg.norm(A @ y - b))

        # project y + q onto A x = b when needed
        if equality_error > delta:
            affine_input = y + q

            x = solve_projection_system(
                affine_input,
                A,
                b,
                x_check,
            )

            # update the second Dykstra correction vector
            q = y + q - x

        # calculate the authors' no-progress measure
        progress_solution = float(np.linalg.norm(x - x_previous))

        # stop when there is no sufficient progress
        if progress_solution <= delta:
            status = "no sufficient progress"
            return x, iteration, status

        # apply the authors' combined error stopping rule
        if bound_error + equality_error <= 2.0 * delta:
            status = "small total error"
            return x, iteration, status

    return x, max_iter, status

In [3]:
def timed_solve(A, b, c, lb, ub, method, crossover=-1, model_name="qr"):
    """Time construction + optimize + extraction; dispose outside that timing.

    optimize_wall_s includes Gurobi's pending model update, presolve, algorithm,
    and crossover when enabled. runtime_s is Gurobi Runtime, also wall time.
    It is not CPU time or a separate crossover measurement.
    """
    start = time.perf_counter()
    model = gp.Model(model_name)
    try:
        model.Params.OutputFlag = OUTPUT_FLAG
        model.Params.Method = method
        model.Params.Crossover = crossover
        model.Params.Threads = THREADS
        model.Params.Seed = SOLVER_SEED
        x = model.addMVar(shape=A.shape[1], lb=lb, ub=ub, name="x")
        model.setObjective(c @ x, GRB.MINIMIZE)
        model.addConstr(A @ x == b, name="equality_constraints")
        before_optimize = time.perf_counter()
        model.optimize()
        after_optimize = time.perf_counter()
        status = model.Status
        solution = x.X.copy() if model.SolCount else None
        objective = model.ObjVal if model.SolCount else np.nan
        total_wall_s = time.perf_counter() - start
        # Read diagnostic attributes after the benchmark timer has stopped.
        # -1 means no concurrent winner was reported, not that Method=-1 won.
        try:
            winner_code = int(model.ConcurrentWinMethod)
        except (gp.GurobiError, AttributeError):
            winner_code = None
        winner_name = {
            0: "Primal simplex", 1: "Dual simplex", 2: "Barrier",
            -1: "No concurrent winner reported",
        }.get(winner_code, "Unavailable")
        info = dict(
            status="OPTIMAL" if status == GRB.OPTIMAL else f"status {status}",
            status_code=status,
            concurrent_win_method=winner_code,
            concurrent_winner=winner_name,
            objective=objective,
            setup_wall_s=before_optimize - start,
            optimize_wall_s=after_optimize - before_optimize,
            total_wall_s=total_wall_s,
            runtime_s=model.Runtime,
            work=model.Work,
            simplex_iterations=model.IterCount,
            barrier_iterations=model.BarIterCount,
            constraint_violation=model.ConstrVio if solution is not None else np.nan,
            bound_violation=model.BoundVio if solution is not None else np.nan,
        )
        return solution, info
    finally:
        model.dispose()


# Initialize the default license/environment once outside all measured solves.
# This excludes one-time license startup fairly from every algorithm and the
# projected solve. No LP is optimized here and no basis is reused.
startup_begin = time.perf_counter()
with gp.Model("initialize_environment") as startup_model:
    solver_defaults = {
        name: getattr(startup_model.Params, name)
        for name in ("Presolve", "FeasibilityTol", "OptimalityTol", "BarConvTol")
    }
startup_wall_s = time.perf_counter() - startup_begin
environment = dict(
    timestamp=datetime.now().astimezone().isoformat(),
    python=sys.version.split()[0], gurobi=".".join(map(str, gp.gurobi.version())),
    numpy=np.__version__, scipy=scipy.__version__,
    os=platform.platform(), cpu=platform.processor(), logical_cpus=os.cpu_count(),
    threads_parameter=THREADS, solver_seed=SOLVER_SEED,
    output_flag=OUTPUT_FLAG, excluded_license_startup_s=startup_wall_s,
    **solver_defaults,
)
# Full environment details remain in `environment`.


Set parameter Username
Set parameter LicenseID to value 2871710
Academic license - for non-commercial use only - expires 2027-10-05


In [4]:
# Shared instance: identical for the original baseline and all projected methods.
D, b = generate_quantile_instance(M, P, DENSITY, DATA_SEED)
A, c = build_quantile_lp(D, b, TAU)
lb, ub = build_quantile_bounds(M, P)
k = int(round((1.0 / EPSILON) ** 2) * np.log(float(A.shape[1])))
if not 0 < k < M:
    raise ValueError(f"Invalid projected dimension k={k}, m={M}")

print("===== Experiment configuration =====")
print(f"Python {environment['python']} | Gurobi {environment['gurobi']}")
print(f"NumPy {environment['numpy']} | SciPy {environment['scipy']} | {environment['os']}")
print(f"CPU: {environment['cpu']} | Logical CPUs: {environment['logical_cpus']}")
print(f"Threads={THREADS} (0=automatic) | Solver seed={SOLVER_SEED} | Runs per setting={REPEATS}")
print(f"m={M}, p={P}, n={A.shape[1]}, epsilon={EPSILON}, k={k}, k/m={k/M:.4f}")
print(f"Data seed={DATA_SEED}, projection seed={PROJECTION_SEED}, tau={TAU}, density={DENSITY}")
if REPEATS == 1:
    print("Single-run timings; timing standard deviations are unavailable.")

# Original LP: Automatic only, fresh model each time, no warm start.
baseline_rows = []
for repeat in range(REPEATS):
    print(f"Solving fixed original Automatic baseline ({repeat + 1}/{REPEATS})...", flush=True)
    row = dict(repeat=repeat + 1, status="NOT RUN", feasible=False,
               objective=np.nan, total_wall_s=np.nan, equality_l2=np.nan,
               max_bound_violation=np.nan, combined_error=np.nan)
    try:
        x_original, info = timed_solve(A, b, c, lb, ub, method=-1,
            crossover=-1, model_name=f"original_automatic_r{repeat + 1}")
        row.update(info)
        if x_original is not None:
            _, row["equality_l2"], row["max_bound_violation"] = evaluate_retrieval(
                x_original, A, b, c, lb, ub)
            row["combined_error"] = bound_error_sum(x_original, lb, ub) + row["equality_l2"]
            row["feasible"] = retrieval_feasibility(x_original, A, b, lb, ub, tol=DELTA)
    except gp.GurobiError as error:
        row["status"] = f"Gurobi error {error.errno}: {error.message}"
    baseline_rows.append(row)
baseline_results = pd.DataFrame(baseline_rows)
valid_baseline = baseline_results[(baseline_results.status == "OPTIMAL") & baseline_results.feasible]
if valid_baseline.empty:
    raise RuntimeError("Original Automatic baseline has no optimal, feasible solve; inspect baseline_results.")
original_time = float(valid_baseline.total_wall_s.mean())
original_objective = float(valid_baseline.objective.mean())
original_time_sd = valid_baseline.total_wall_s.std(ddof=1) if len(valid_baseline) > 1 else np.nan
if len(valid_baseline) != REPEATS:
    print(f"Incomplete original baseline: {len(valid_baseline)}/{REPEATS} valid runs; means use valid runs only.")
if not np.allclose(valid_baseline.objective, original_objective, rtol=1e-6, atol=1e-7):
    raise RuntimeError("Original objective values disagree across valid baseline solves.")

print("\n===== Fixed original-LP baseline =====")
print(f"Optimal and feasible runs: {len(valid_baseline)}/{REPEATS}")
print(f"Original LP time:          {original_time:.3f} s" + (" (mean)" if REPEATS > 1 else " (one run)"))
print("Time standard deviation: " + (f"{original_time_sd:.3f} s" if np.isfinite(original_time_sd) else "n/a (one run)"))
print(f"Original objective:        {original_objective:.6f}")
print(f"Worst equality L2:         {valid_baseline.equality_l2.max():.3e}")
print(f"Worst bound violation:     {valid_baseline.max_bound_violation.max():.3e}")
print(f"Worst combined error:      {valid_baseline.combined_error.max():.3e}")
if "concurrent_winner" in valid_baseline:
    print("Automatic winner(s):      " + ", ".join(f"{name} ({count})" for name, count in valid_baseline.concurrent_winner.value_counts().items()))

# Fixed T and projected system: generated once, shared by every projected run.
start = time.perf_counter()
T = generate_projection_matrix(A, k, PROJECTION_SEED)
time_T = time.perf_counter() - start
start = time.perf_counter()
A_projected, b_projected = project_lp(A, b, T)
time_projection = time.perf_counter() - start
shared_projection_time = time_T + time_projection
print("\n===== Shared projection work (seconds; performed once) =====")
print(f"Generate T:           {time_T:.3f}")
print(f"Compute TA and Tb:    {time_projection:.3f}")
print(f"Shared cost:          {shared_projection_time:.3f}")


===== Experiment configuration =====
Python 3.14.7 | Gurobi 13.0.3
NumPy 2.5.3 | SciPy 1.18.1 | Windows-11-10.0.26300-SP0
CPU: AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD | Logical CPUs: 16
Threads=0 (0=automatic) | Solver seed=0 | Runs per setting=1
m=5000, p=399, n=10399, epsilon=0.2, k=231, k/m=0.0462
Data seed=42, projection seed=42, tau=0.2, density=0.8
Single-run timings; timing standard deviations are unavailable.
Solving fixed original Automatic baseline (1/1)...

===== Fixed original-LP baseline =====
Optimal and feasible runs: 1/1
Original LP time:          8.710 s (one run)
Time standard deviation: n/a (one run)
Original objective:        1123.037359
Worst equality L2:         9.479e-14
Worst bound violation:     0.000e+00
Worst combined error:      9.479e-14
Automatic winner(s):      Barrier (1)

===== Shared projection work (seconds; performed once) =====
Generate T:           0.005
Compute TA and Tb:    0.028
Shared cost:          0.032


In [5]:
# Projected LP: vary only the Gurobi algorithm/crossover setting.
# Each projected solution receives its own retrieval; no solution is reused.
rows = []
for repeat in range(REPEATS):
    shift = repeat % len(ALGORITHMS)
    order = ALGORITHMS[shift:] + ALGORITHMS[:shift]
    for position, (name, method, crossover) in enumerate(order, start=1):
        print(f"Solving projected {name} ({repeat + 1}/{REPEATS})...", flush=True)
        row = dict(algorithm=name, method=method, crossover=crossover,
            repeat=repeat + 1, order_position=position, status="NOT RUN",
            objective=np.nan, setup_wall_s=np.nan, optimize_wall_s=np.nan,
            total_wall_s=np.nan, simplex_iterations=np.nan, barrier_iterations=np.nan,
            concurrent_winner="Unavailable", projected_equality_l2=np.nan,
            projected_max_bound_violation=np.nan, retrieval_completed=False,
            retrieval_status="not run", retrieval_wall_s=np.nan,
            pipeline_wall_s=np.nan, dykstra_iterations=np.nan,
            retrieved_objective=np.nan, relative_objective_difference=np.nan,
            feasible_objective_gap=np.nan, equality_l2=np.nan,
            max_bound_violation=np.nan, bound_violation_sum=np.nan,
            combined_error=np.nan, avgin=np.nan, avgeq=np.nan, feasible=False,
            runtime_ratio=np.nan, speedup=np.nan)
        try:
            x_projected, info = timed_solve(A_projected, b_projected, c, lb, ub,
                method, crossover, model_name=f"projected_r{repeat + 1}_m{method}_c{crossover}")
            row.update(info)
        except gp.GurobiError as error:
            row["status"] = f"Gurobi error {error.errno}: {error.message}"
            rows.append(row)
            print(row["status"])
            continue
        if info["status_code"] != GRB.OPTIMAL:
            rows.append(row)
            print(f"Projected solve failed: {info['status']}; retrieval skipped.")
            continue
        # All quality checks and winner reporting are outside solve/retrieval timers.
        _, row["projected_equality_l2"], row["projected_max_bound_violation"] = evaluate_retrieval(
            x_projected, A_projected, b_projected, c, lb, ub)
        if method == -1:
            print(f"Projected Automatic concurrent winner: {info['concurrent_winner']}")
        start = time.perf_counter()
        try:
            x_retrieved, iterations, status = retrieve_dykstra(x_projected, A, b,
                lb, ub, max_iter=MAX_ITER, delta=DELTA)
        except Exception as error:
            row["retrieval_wall_s"] = time.perf_counter() - start
            row["retrieval_status"] = f"Retrieval error: {type(error).__name__}: {error}"
            rows.append(row)
            print(row["retrieval_status"])
            continue
        row["retrieval_wall_s"] = time.perf_counter() - start
        row["pipeline_wall_s"] = shared_projection_time + info["total_wall_s"] + row["retrieval_wall_s"]
        row["retrieval_completed"] = True
        row["dykstra_iterations"], row["retrieval_status"] = iterations, status
        row["retrieved_objective"], row["equality_l2"], row["max_bound_violation"] = evaluate_retrieval(
            x_retrieved, A, b, c, lb, ub)
        row["bound_violation_sum"] = bound_error_sum(x_retrieved, lb, ub)
        row["combined_error"] = row["bound_violation_sum"] + row["equality_l2"]
        row["avgin"], row["avgeq"] = evaluate_errors(x_retrieved, A, b, M, P)
        row["feasible"] = retrieval_feasibility(x_retrieved, A, b, lb, ub, tol=DELTA)
        if abs(original_objective) > 1e-12:
            row["relative_objective_difference"] = abs(row["retrieved_objective"] - original_objective) / abs(original_objective)
            if row["feasible"]:
                row["feasible_objective_gap"] = row["relative_objective_difference"]
        row["runtime_ratio"] = row["pipeline_wall_s"] / original_time
        row["speedup"] = original_time / row["pipeline_wall_s"]
        rows.append(row)
results = pd.DataFrame(rows)


Solving projected Automatic (1/1)...
Projected Automatic concurrent winner: Dual simplex
Solving projected Primal simplex (1/1)...
Solving projected Dual simplex (1/1)...
Solving projected Barrier default crossover (1/1)...
Solving projected Barrier without crossover (1/1)...


In [6]:
# Detailed results: all metrics refer to each method's own retrieved point.
for name, method, crossover in ALGORITHMS:
    group = results[results.algorithm == name]
    print(f"\n===== Projected {name} =====")
    for _, run in group.iterrows():
        print(f"Run {int(run['repeat'])}/{REPEATS}: {run['status']}")
        if run["status"] != "OPTIMAL":
            continue
        print(f"Projected model setup:    {run['setup_wall_s']:.3f} s")
        print(f"Projected optimization:   {run['optimize_wall_s']:.3f} s")
        print(f"Projected solve total:    {run['total_wall_s']:.3f} s")
        print(f"Projected objective:      {run['objective']:.6f}")
        print(f"Projected equality L2:    {run['projected_equality_l2']:.3e}")
        print(f"Projected max bound error: {run['projected_max_bound_violation']:.3e}")
        print(f"Simplex / barrier iterations: {run['simplex_iterations']:.0f} / {run['barrier_iterations']:.0f}")
        if method == -1:
            print(f"Concurrent winner:        {run['concurrent_winner']}")
        if not run["retrieval_completed"]:
            print(run["retrieval_status"])
            continue
        print(f"Dykstra retrieval:        {run['retrieval_wall_s']:.3f} s")
        print(f"Complete RP pipeline:     {run['pipeline_wall_s']:.3f} s")
        print(f"Original objective:       {original_objective:.6f}")
        print(f"Retrieved objective:      {run['retrieved_objective']:.6f}")
        print(f"Absolute objective difference: {100*run['relative_objective_difference']:.3f}% (descriptive)")
        print("Feasible objective gap:   " + (f"{100*run['feasible_objective_gap']:.3f}%" if np.isfinite(run['feasible_objective_gap']) else "n/a"))
        print(f"Equality residual (L2):   {run['equality_l2']:.3e}")
        print(f"Maximum bound violation:  {run['max_bound_violation']:.3e}")
        print(f"Sum of bound violations:  {run['bound_violation_sum']:.6f}")
        print(f"Average inequality error: {run['avgin']:.3e}")
        print(f"Average equality error:   {run['avgeq']:.3e}")
        print(f"Combined error:           {run['combined_error']:.6f} (required <= {2*DELTA:.3f})")
        print(f"Feasible at tolerance:    {run['feasible']}")
        print(f"Dykstra iterations / stop: {int(run['dykstra_iterations'])} / {run['retrieval_status']}")
        print(f"RP / original time ratio: {run['runtime_ratio']:.3f}")
        print(f"Direct / RP pipeline:     {run['speedup']:.3f}x")
        if not run["feasible"]:
            print("Infeasible retrieved point: timing observation only, not a valid feasible-solution success.")



===== Projected Automatic =====
Run 1/1: OPTIMAL
Projected model setup:    0.040 s
Projected optimization:   0.380 s
Projected solve total:    0.424 s
Projected objective:      0.000000
Projected equality L2:    8.025e-11
Projected max bound error: 0.000e+00
Simplex / barrier iterations: 231 / 0
Concurrent winner:        Dual simplex
Dykstra retrieval:        1.333 s
Complete RP pipeline:     1.790 s
Original objective:       1123.037359
Retrieved objective:      1246.671845
Absolute objective difference: 11.009% (descriptive)
Feasible objective gap:   n/a
Equality residual (L2):   2.110e-03
Maximum bound violation:  3.412e-04
Sum of bound violations:  0.557745
Average inequality error: 1.115e-04
Average equality error:   2.379e-05
Combined error:           0.559855 (required <= 0.020)
Feasible at tolerance:    False
Dykstra iterations / stop: 12 / no sufficient progress
RP / original time ratio: 0.205
Direct / RP pipeline:     4.866x
Infeasible retrieved point: timing observation onl

In [7]:
from IPython.display import display

# Aggregate only successful solves/completed retrievals; count feasible successes separately.
summary_rows = []
optimization_rows = []
for name, method, crossover in ALGORITHMS:
    group = results[results.algorithm == name]
    optimal = group[group.status == "OPTIMAL"]
    completed = optimal[optimal.retrieval_completed]
    feasible = completed[completed.feasible]
    def mean_of(frame, column):
        return float(frame[column].mean()) if len(frame) else np.nan
    mean_pipeline = mean_of(completed, "pipeline_wall_s")
    summary_rows.append(dict(algorithm=name, optimal_runs=len(optimal),
        completed_runs=len(completed), feasible_runs=len(feasible),
        mean_solve_s=mean_of(optimal, "total_wall_s"),
        mean_retrieval_s=mean_of(completed, "retrieval_wall_s"),
        mean_pipeline_s=mean_pipeline,
        sd_pipeline_s=float(completed.pipeline_wall_s.std(ddof=1)) if len(completed) > 1 else np.nan,
        ratio=mean_pipeline/original_time,
        speedup=original_time/mean_pipeline if mean_pipeline > 0 else np.nan,
        # A summary gap is valid only when every attempted run returned a feasible point.
        gap_percent=100*mean_of(feasible, "feasible_objective_gap") if len(feasible) == REPEATS else np.nan,
        feasibility_error=float(completed.combined_error.max()) if len(completed) else np.nan))
    winners = ", ".join(f"{winner} ({count})" for winner, count in optimal.concurrent_winner.value_counts().items()) if method == -1 else "Not applicable"
    optimization_rows.append({"Algorithm": name, "Optimal runs": f"{len(optimal)}/{REPEATS}",
        "Setup (s)": mean_of(optimal, "setup_wall_s"),
        "Optimize (s)": mean_of(optimal, "optimize_wall_s"),
        "Simplex iterations": mean_of(optimal, "simplex_iterations"),
        "Barrier iterations": mean_of(optimal, "barrier_iterations"),
        "Concurrent winner(s)": winners})
summary = pd.DataFrame(summary_rows)
summary_table = pd.DataFrame({
    "TLP algorithm": summary.algorithm,
    "Original LP Automatic (s)": original_time,
    "Optimal / feasible runs": [f"{int(r.optimal_runs)}/{REPEATS} | {int(r.feasible_runs)}/{REPEATS}" for _, r in summary.iterrows()],
    "TLP solve (s)": summary.mean_solve_s,
    "Retrieval (s)": summary.mean_retrieval_s,
    "Total RP (s)": summary.mean_pipeline_s,
    "SD total (s)": summary.sd_pipeline_s,
    "RP / original": summary.ratio,
    "Speedup": summary.speedup,
    "Retrieved gap (%)": summary.gap_percent,
    "Feasibility error": summary.feasibility_error,
})
optimization_table = pd.DataFrame(optimization_rows)
print("\n===== Complete projected pipeline comparison =====")
print("Times are single observations." if REPEATS == 1 else "Times are arithmetic means over successful/completed runs.")
print("Original LP Automatic (s) is the same fixed baseline in every row.")
display(summary_table.style.hide(axis="index").format({
    "Original LP Automatic (s)": "{:.3f}", "TLP solve (s)": "{:.3f}", "Retrieval (s)": "{:.3f}", "Total RP (s)": "{:.3f}",
    "SD total (s)": "{:.3f}", "RP / original": "{:.3f}", "Speedup": "{:.3f}x",
    "Retrieved gap (%)": "{:.3f}", "Feasibility error": "{:.6f}",
}, na_rep="n/a"))
print(f"Feasibility error = sum of bound violations + equality L2; required <= {2*DELTA:.3f}.")
print("Retrieved gap is n/a for infeasible/incomplete pipelines or a zero original objective.")
print("Optimal / feasible runs counts projected LP optimal solves and original-feasible retrieved points separately.")
print("\n===== Projected optimization-only comparison =====")
display(optimization_table.style.hide(axis="index").format({
    "Setup (s)": "{:.3f}", "Optimize (s)": "{:.3f}",
    "Simplex iterations": "{:.0f}", "Barrier iterations": "{:.0f}",
}, na_rep="n/a"))
print("Optimize measures the optimize() call, including pending updates, presolve, and crossover when enabled.")



===== Complete projected pipeline comparison =====
Times are single observations.


TLP algorithm,Original LP Automatic (s),Optimal / feasible runs,TLP solve (s),Retrieval (s),Total RP (s),SD total (s),RP / original,Speedup,Retrieved gap (%),Feasibility error
Automatic,8.710,1/1 | 0/1,0.424,1.333,1.790,n/a,0.205,4.866x,n/a,0.559855
Primal simplex,8.710,1/1 | 0/1,0.473,1.316,1.822,n/a,0.209,4.781x,n/a,0.573214
Dual simplex,8.710,1/1 | 0/1,0.321,1.326,1.679,n/a,0.193,5.188x,n/a,0.559855
Barrier default crossover,8.710,1/1 | 0/1,0.405,1.295,1.732,n/a,0.199,5.029x,n/a,0.573158
Barrier without crossover,8.710,1/1 | 0/1,0.249,1.195,1.476,n/a,0.169,5.901x,n/a,0.573158


Feasibility error = sum of bound violations + equality L2; required <= 0.020.
Retrieved gap is n/a for infeasible/incomplete pipelines or a zero original objective.
Optimal / feasible runs counts projected LP optimal solves and original-feasible retrieved points separately.

===== Projected optimization-only comparison =====


Algorithm,Optimal runs,Setup (s),Optimize (s),Simplex iterations,Barrier iterations,Concurrent winner(s)
Automatic,1/1,0.040,0.380,231,0,Dual simplex (1)
Primal simplex,1/1,0.038,0.431,231,0,Not applicable
Dual simplex,1/1,0.032,0.286,231,0,Not applicable
Barrier default crossover,1/1,0.029,0.373,171,5,Not applicable
Barrier without crossover,1/1,0.036,0.209,0,5,Not applicable


Optimize measures the optimize() call, including pending updates, presolve, and crossover when enabled.


In [8]:
# Interpretation derived from the measurements, not from a predetermined winner.
print("\n===== Interpretation =====")
complete_solver = optimization_table[summary.optimal_runs.to_numpy() == REPEATS]
complete_pipeline = summary[summary.completed_runs == REPEATS]
if len(complete_solver):
    fastest_solver = complete_solver.loc[complete_solver["Optimize (s)"].idxmin()]
    print(f"Fastest projected optimization: {fastest_solver['Algorithm']} ({fastest_solver['Optimize (s)']:.3f} s).")
    automatic = complete_solver[complete_solver.Algorithm == "Automatic"]
    if len(automatic) and fastest_solver["Optimize (s)"] > 0:
        factor = float(automatic["Optimize (s)"].iloc[0]) / fastest_solver["Optimize (s)"]
        print(f"Automatic optimization took {factor:.3f}x the fastest measured optimization time.")
if len(complete_pipeline):
    fastest_pipeline = complete_pipeline.loc[complete_pipeline.mean_pipeline_s.idxmin()]
    print(f"Fastest complete computational pipeline: {fastest_pipeline.algorithm} ({fastest_pipeline.mean_pipeline_s:.3f} s).")
    print(f"Across complete pipelines, RP/original ratios range from {complete_pipeline.ratio.min():.3f} to {complete_pipeline.ratio.max():.3f}.")
    print("The original baseline is fixed; these ratio changes reflect projected solve plus retrieval time changes.")
fully_feasible = complete_pipeline[complete_pipeline.feasible_runs == REPEATS]
if len(fully_feasible):
    best_feasible = fully_feasible.loc[fully_feasible.mean_pipeline_s.idxmin()]
    print(f"Fastest pipeline passing the reference feasibility criterion on every run: {best_feasible.algorithm} ({best_feasible.speedup:.3f}x computational speedup).")
else:
    print("No configuration returned feasible retrieved points on every run; no complete feasible-workflow success is established.")
completed_runs = results[results.retrieval_completed]
if len(completed_runs):
    print(f"Retrieved objective range: {completed_runs.retrieved_objective.min():.6f} to {completed_runs.retrieved_objective.max():.6f}.")
    print(f"Combined feasibility error range: {completed_runs.combined_error.min():.6f} to {completed_runs.combined_error.max():.6f}; required <= {2*DELTA:.3f}.")
    for _, r in summary.iterrows():
        print(f"{r.algorithm}: {int(r.feasible_runs)}/{REPEATS} feasible retrieved points.")
if any(summary.optimal_runs != REPEATS) or any(summary.completed_runs != REPEATS):
    print("Failures occurred; inspect results and do not rank partial averages as complete successes.")
print(f"Liberti Table 5 (m=5000, epsilon=0.2) reports RP/original time ratio {PAPER_RATIO:.2f}.")
print("Liberti used HiGHS revised simplex for projected QR and CPLEX Automatic for the original LP.")
print("This experiment uses Gurobi for both; solver implementation, hardware, timing scope, and retrieval differences prevent direct replication.")
if REPEATS == 1:
    print("One run per setting: observed differences are preliminary, with no timing-variability estimate.")
print("The existing Dykstra early-return issue is retained; feasibility is checked on the actual returned point.")



===== Interpretation =====
Fastest projected optimization: Barrier without crossover (0.209 s).
Automatic optimization took 1.820x the fastest measured optimization time.
Fastest complete computational pipeline: Barrier without crossover (1.476 s).
Across complete pipelines, RP/original ratios range from 0.169 to 0.209.
The original baseline is fixed; these ratio changes reflect projected solve plus retrieval time changes.
No configuration returned feasible retrieved points on every run; no complete feasible-workflow success is established.
Retrieved objective range: 1183.140196 to 1246.671845.
Combined feasibility error range: 0.559855 to 0.573214; required <= 0.020.
Automatic: 0/1 feasible retrieved points.
Primal simplex: 0/1 feasible retrieved points.
Dual simplex: 0/1 feasible retrieved points.
Barrier default crossover: 0/1 feasible retrieved points.
Barrier without crossover: 0/1 feasible retrieved points.
Liberti Table 5 (m=5000, epsilon=0.2) reports RP/original time ratio 0.1